In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d

In [ ]:
col_headers = ['alpha','cl','cd','cdp','cm','topxtr','botxtr']
df_50 = pd.read_csv(r'C:\Users\mayar\Documents\Ryerson\AALOFT\George Airfoil\mh32_re50k.txt', skiprows=9, header=1, sep=r'\s+', names=col_headers)
df_100 = pd.read_csv(r'C:\Users\mayar\Documents\Ryerson\AALOFT\George Airfoil\mh32_re100k.txt', skiprows=9, header=1, sep=r'\s+', names=col_headers)
df_200 = pd.read_csv(r'C:\Users\mayar\Documents\Ryerson\AALOFT\George Airfoil\mh32_re200k.txt', skiprows=9, header=1, sep=r'\s+', names=col_headers)
df_500 = pd.read_csv(r'C:\Users\mayar\Documents\Ryerson\AALOFT\George Airfoil\mh32_re500k.txt', skiprows=9, header=1, sep=r'\s+', names=col_headers)

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2,2,figsize=(10,10))

ax1.plot(df_50.cd, df_50.cl, color='tab:orange', label='Re 50k')
ax1.plot(df_100.cd, df_100.cl, color='tab:blue', label='Re 100k')
ax1.plot(df_200.cd, df_200.cl, color='tab:green', label='Re 200k')
ax1.plot(df_500.cd, df_500.cl, color='tab:red', label='Re 500k')
ax1.set_xlabel('Cd')
ax1.set_ylabel('Cl')
ax1.grid()
ax1.legend()

ax2.plot(df_50.alpha, df_50.cl, color='tab:orange', label='Re 50k')
ax2.plot(df_100.alpha, df_100.cl, color='tab:blue', label='Re 100k')
ax2.plot(df_200.alpha, df_200.cl, color='tab:green', label='Re 200k')
ax2.plot(df_500.alpha, df_500.cl, color='tab:red', label='Re 500k')
ax2.set_xlabel('Alpha')
ax2.set_ylabel('Cl')
ax2.grid()
ax2.legend()

ax3.plot(df_50.alpha, df_50.cd, color='tab:orange', label='Re 50k')
ax3.plot(df_100.alpha, df_100.cd, color='tab:blue', label='Re 100k')
ax3.plot(df_200.alpha, df_200.cd, color='tab:green', label='Re 200k')
ax3.plot(df_500.alpha, df_500.cd, color='tab:red', label='Re 500k')
ax3.set_xlabel('Alpha')
ax3.set_ylabel('Cd')
ax3.grid()
ax3.legend()

ax4.plot(df_50.alpha, (df_50.cl)/(df_50.cd), color='tab:orange', label='Re 50k')
ax4.plot(df_100.alpha, (df_100.cl)/(df_100.cd), color='tab:blue', label='Re 100k')
ax4.plot(df_200.alpha, (df_200.cl)/(df_200.cd), color='tab:green', label='Re 200k')
ax4.plot(df_500.alpha, (df_500.cl)/(df_500.cd), color='tab:red', label='Re 500k')
ax4.set_xlabel('Alpha')
ax4.set_ylabel('Cl/Cd')
ax4.grid()
ax4.legend()
plt.show()

In [ ]:
col_headers = ['x','y']
mh32_airfoil = pd.read_csv(r'C:\Users\mayar\Documents\Ryerson\Grad Classes\AE8139 MDO\MH32.dat', skiprows=1, sep=r'\s+', names=col_headers)

In [ ]:
fig, (ax1) = plt.subplots(1,1,figsize=(10,1.25))

ax1.plot(mh32_airfoil.x, mh32_airfoil.y, color='tab:blue')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.grid()

plt.show()

## Testing of Aero Code

Pseudo-Code:
1. Calculate initial performance of wing
2. Find the speed for max range using power-required vs. airspeed
3. Find the aoa corresponding to speed for max range
4. Re-run FW at specific aoa for max range
5. Find lift-distribution at specified aoa
6. Restart from 1. with tip deflections

In [2]:
from src.main import aero, mass, wing_area
from src.freewake_parse import freewake_input, freewake_run

wingspan = 2.00
mid_chord = 0.15
tip_chord = 0.15
root_twist = 0
mid_twist = 0
tip_twist = 0
spar_mat = 'pla'
skin_mat = 'pla'
spar_volume = 0.3*mid_chord*wingspan*0.01
skin_volume = wingspan*mid_chord*2*0.001
deflect_tip = 0
deflect_mid = 0

# V_maxR, Preq, y_pos, y_load = aero(2.00, 0.15, 0.15, 0, 0, 0, 'pla', 'pla', 0.15, 0.05, 0, 0)


In [ ]:
# CONSTANTS
g = 9.81

# Aircraft sizing
S = wing_area(wingspan, mid_chord, tip_chord)
AR = (wingspan**2)/S

# Aircraft weight
m = mass(spar_mat, skin_mat, spar_volume, skin_volume)
W = m*g

# Create input file for FreeWake
freewake_input(wingspan, mid_chord, tip_chord, root_twist, mid_twist, tip_twist, W, deflect_tip, deflect_mid)
df_fw_init, _ = freewake_run()

# # Fit second-order curve to airspeed vs. power
# coefficients, _ = curve_fit(power_eqn, df_fw_init['Vinf'], df_fw_init['Preq'])
# a_fit, b_fit, c_fit = coefficients

# # Find speed for max range
# V_maxR = np.sqrt(c_fit/a_fit)

# # # Find power required at speed for max range
# Preq = power_eqn(V_maxR, a_fit, b_fit, c_fit)

# # Find aoa for max range
# # CL = W/(0.5*1.225*(V_maxR**2)*wingspan)
# # CL_alpha = (2*np.pi)*(AR/(AR+2))
# # aoa_maxR = np.rad2deg(CL/CL_alpha)
# f_aoa_max = interp1d(df_fw_init['Vinf'],df_fw_init['alpha'],kind='linear')
# aoa_maxR = f_aoa_max(V_maxR)

# freewake_input(wingspan, mid_chord, tip_chord, root_twist, mid_twist, tip_twist, W, deflect_tip, deflect_mid, aoa_maxR, aoa_maxR, 1)
# _, df_load = freewake_run(aoa_maxR)

# y_pos = df_load['yo']
# y_load = 0.5*1.225*V_maxR*V_maxR*df_load['S']*df_load['cl']

In [ ]:
aoa_filepath = r'C:\Users\mayar\Documents\Ryerson\AALOFT\Freewake(in_use)\output\AOA5.50.txt'
perf_filepath = r'C:\Users\mayar\Documents\Ryerson\AALOFT\Freewake(in_use)\output\performance.txt'

aoa_col = ['index','xo','yo','zo','cn','cl','cy','cd','A','B','C','S','span','chord','nu','epsilon','psi','phiLE','#']
aoa = pd.read_csv(aoa_filepath, skiprows=4, sep=r'\s+', nrows=40, header=None, names=aoa_col)
performance = pd.read_csv(perf_filepath, skiprows=3, sep=r'\s+')
# performance = performance.drop([0])

def power_eqn(V, a, b, c):
    return np.array(a*(V**2) + b*V + c)


In [ ]:
# Fit second-order curve to airspeed vs. power
coefficients, _ = curve_fit(power_eqn, performance['Vinf'], performance['Preq'])
a_fit, b_fit, c_fit = coefficients

# Find speed for max range
V_maxR = np.sqrt(c_fit/a_fit)

# Find power required at speed for max range
Preq = power_eqn(V_maxR, a_fit, b_fit, c_fit)

f_aoa_max = interp1d(performance['Vinf'],performance['alpha'],kind='linear')
aoa_max = f_aoa_max(V_maxR)

# V_maxR = 19
CL = 245.25/(0.5*1.225*(V_maxR**2)*2.1)

AR = (5.25**2)/2.10
# CL_alpha = (np.pi*AR)/(np.sqrt(1+(np.sqrt(1+((AR/2)**2)))))
CL_alpha = (2*np.pi)*(AR/(AR+2))

alpha_0 = -5.5
aoa_calc = np.rad2deg(CL/CL_alpha)


In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(15,5))

ax1.scatter(performance['Vinf'],performance['Preq'])
ax1.plot(np.linspace(min(performance['Vinf']),max(performance['Vinf']),50),power_eqn(np.linspace(min(performance['Vinf']),max(performance['Vinf']),50),a_fit,b_fit,c_fit),color='tab:red')
ax1.scatter(V_maxR,Preq,marker='x')
ax1.set_xlabel('Airspeed (m/s)')
ax1.set_ylabel('Power Required (W)')
ax1.grid()

ax2.scatter(performance['alpha'],performance['Vinf'])
ax2.plot(np.linspace(0,10,10),np.ones(10)*V_maxR,color='black')
ax2.plot(aoa_max,V_maxR,marker='x',color='tab:red')
ax2.plot(aoa_calc,V_maxR,marker='x',color='tab:green')
ax2.set_xlabel('Alpha')
ax2.set_ylabel('Vinf')
ax2.grid()

plt.show()

In [ ]:
load = 0.5*1.225*performance['Vinf'][10]*performance['Vinf'][10]*aoa['S']*aoa['cl']

fig, (ax1, ax2) = plt.subplots(1,2,figsize=(15,5))

ax1.scatter(aoa['yo'],aoa['cl'])
ax1.set_xlabel('Y-position')
ax1.set_ylabel('Section cl')
ax1.grid()

ax2.scatter(aoa['yo'],load)
ax2.set_xlabel('Y-position')
ax2.set_ylabel('Load (N)')
ax2.grid()

plt.show()

In [ ]:
cumulative_trapezoid(aoa['cl'],aoa['yo'])
